# Visualise clusters on map

*Author: loua*

## Setup

In [1]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle

mpl.use("pgf")
mpl.rcParams.update({
    "pgf.rcfonts": False,      # don't override LaTeX fonts
    "text.usetex": True,       # use LaTeX for all text
    "font.family": "serif",    # match LaTeX document font
    "font.size": 20,
    "axes.titlesize": 24,
    "axes.labelsize": 16,
    "axes.labelweight": "normal",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "grid.linestyle": "--",
    "grid.alpha": 0.3,
})

COLOURS = [
    "#2580B7", # Blue
    "#179E86", # Turquoise
    "#9EBE5B", # Light Green
    "#F59B11", # Yellow
    "#F24D00", # Orange
]

In [2]:
ROOT = '../../'
PATH = ROOT + 'processed-data/'
CLUSTERS = PATH + 'characterising-neighbourhoods/05_clusters.csv'
ORIGINAL = PATH + '2021-special/demo_relative_2021.csv'
ABSOLUTE = PATH + '2021-special/demo_absolute_2021.csv'
CLEANED = PATH + 'characterising-neighbourhoods/02_afterEDA.csv'
GEO = PATH + 'geodata/afstemningsomraader2021_CPH_FRB.geojson'
OUTPUT = PATH + 'characterising-neighbourhoods/06_demographics'
PLOT_PATH = ROOT + 'data-wrangling/characterising-neighbourhoods/plots/'
OUTPUT_PATH_PNG = PLOT_PATH + 'png/'
OUTPUT_PATH_PGF = PLOT_PATH + 'pgf/'
OUTPUT_PATH_PDF = PLOT_PATH + 'pdf/'

## Load and merge data

In [3]:
original_data = pd.read_csv(ORIGINAL, dtype={'PollingAreaID': str})
absolute_data = pd.read_csv(ABSOLUTE, dtype={'PollingAreaID': str})
cleaned_data = pd.read_csv(CLEANED, dtype={'PollingAreaID': str})
cluster_data = pd.read_csv(CLUSTERS, dtype={'PollingAreaID': str})
geo = gpd.read_file(GEO)

In [4]:
# Create consistent PollingAreaID
geo['PollingAreaID'] = (
    geo['kommunekode'].str[-3:] + 
    '0' +
    geo['afstemningsomraadenummer'].str.zfill(2)
)

# Select columns
id_cols = ['PollingAreaID', 'Gruppe', 'Name', 'DistrictNo', 'District', 'Municipality']
original_subset = original_data[id_cols]

cluster_cols = ['PollingAreaID', 'PollingArea', 'Cluster_kMeans', 'PC1', 'PC2', 'PC3', 
                'PC4', 'PC5', 'Cluster_Hierarchical', 'Cluster_GMM', 'Cluster_DBSCAN', 
                'Outlier_Distance to centre']
cluster_subset = cluster_data[cluster_cols]

population_subset = absolute_data[['PollingAreaID'] + ['Population']]

demographic_cols = [col for col in cleaned_data.columns 
                    if col not in ['PollingAreaID', 'PollingArea']]
demo_subset = cleaned_data[['PollingAreaID'] + demographic_cols]

geo_subset = geo[['PollingAreaID', 'afstemningsstednavn', 'areaSquareMetres', 'geometry']]

gdf = geo_subset.copy()
gdf = gdf.merge(cluster_subset, on='PollingAreaID', how='left')
gdf = gdf.merge(original_subset, on='PollingAreaID', how='left')
gdf = gdf.merge(population_subset, on='PollingAreaID', how='left')
gdf = gdf.merge(demo_subset, on='PollingAreaID', how='left')

id_columns = ['PollingAreaID', 'PollingArea', 'afstemningsstednavn', 'Gruppe', 'Name', 
              'DistrictNo', 'District', 'Municipality']
pca_columns = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
cluster_columns = ['Cluster_kMeans', 'Cluster_Hierarchical', 'Cluster_GMM', 
                   'Cluster_DBSCAN', 'Outlier_Distance to centre']
area_pop_columns = ['areaSquareMetres', 'Population']
all_specified = id_columns + pca_columns + cluster_columns + area_pop_columns + ['geometry']
demo_columns = [col for col in gdf.columns if col not in all_specified]
column_order = (id_columns + pca_columns + cluster_columns + area_pop_columns + demo_columns + ['geometry'])
gdf = gdf[column_order]

print(f"GeoJSON rows: {len(geo)}")
print(f"Merged rows: {len(gdf)}")
print(f"Columns: {len(gdf.columns)}")

gdf.to_file(OUTPUT + '.geojson', 
               driver='GeoJSON')

gdf_csv = gdf.drop(columns='geometry')
gdf_csv.to_csv(OUTPUT + '.csv', 
                  index=False)

GeoJSON rows: 61
Merged rows: 61
Columns: 62


## Create choropleth map

### Interactive

In [5]:
gdf['Cluster_kMeans'] = gdf['Cluster_kMeans'].astype(str)

# Hover text
gdf['hover_text'] = (
    gdf['PollingArea'] + ' (' + gdf['Municipality'] + ').' +
    ' Population: ' + gdf['Population'].astype(str)
)

fig = px.choropleth_map(
    gdf,
    geojson=gdf.geometry.__geo_interface__,
    locations=gdf.index,
    color='Cluster_kMeans',
    color_discrete_sequence=COLOURS,
    map_style="carto-positron",
    center={"lat": 55.68, "lon": 12.57},
    zoom=10,
    opacity=0.8,
    labels={'Cluster_kMeans': 'Cluster'},
    category_orders={'Cluster_kMeans': ['0', '1', '2', '3', '4']},
    hover_name='hover_text',
    hover_data={'Cluster_kMeans': False, 'hover_text': False}
)

fig.update_traces(hovertemplate='%{hovertext}<extra></extra>')

fig.update_layout(
    margin={"r":0,"t":0,"l":0,"b":0},
    showlegend=True
)

fig.show()

### Static

In [6]:
gdf['Cluster_kMeans'] = gdf['Cluster_kMeans'].astype(int)

# Color mapping
color_map = {i: COLOURS[i] for i in range(5)}
gdf['color'] = gdf['Cluster_kMeans'].map(color_map)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
gdf.plot(ax=ax, color=gdf['color'], alpha=0.9, edgecolor='white', linewidth=0.5)
ax.axis('off')

# Legend
legend_elements = [Patch(facecolor=COLOURS[i], label=f'Cluster {i}') 
                   for i in range(5)]
ax.legend(handles=legend_elements, 
          loc='lower right',
          fontsize=16,
          frameon=True,
          markerscale=1.5)

# Crop right side
xlim = ax.get_xlim()
ax.set_xlim(xlim[0], xlim[1] * 0.994)

plt.tight_layout()
filename = 'cluster_map'
plt.savefig(f"{OUTPUT_PATH_PNG}{filename}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{OUTPUT_PATH_PGF}{filename}.pgf")
plt.savefig(f"{OUTPUT_PATH_PDF}{filename}.pdf")
plt.show()

/var/folders/j3/746qr67x2xj8z46fp8cgvc080000gn/T/ipykernel_8992/3898058385.py:30: UserWarning:

FigureCanvasPgf is non-interactive, and thus cannot be shown



In [7]:
# Get Denmark map
denmark_gdf = gpd.read_file(ROOT + 'processed-data/geodata/dagi-500-regioner.geojson')

# Color mapping
color_map = {i: COLOURS[i] for i in range(5)}
gdf['color'] = gdf['Cluster_kMeans'].map(color_map)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
gdf.plot(ax=ax, color=gdf['color'], alpha=0.9, edgecolor='white', linewidth=0.5)
ax.axis('off')

# Legend
legend_elements = [Patch(facecolor=COLOURS[i], label=f'Cluster {i}') 
                   for i in range(5)]
ax.legend(handles=legend_elements, 
          loc='lower right',
          fontsize=16,
          frameon=True,
          markerscale=1.5)

# Crop right side
xlim = ax.get_xlim()
ax.set_xlim(xlim[0], xlim[1] * 0.994)

# Inset map
ax_inset = inset_axes(ax, width="30%", height="30%", loc='lower left',
                      bbox_to_anchor=(0.02, 0.02, 1, 1),
                      bbox_transform=ax.transAxes,
                      borderpad=0)

# Plot all of Denmark
denmark_gdf.plot(ax=ax_inset, color='lightgray', edgecolor='lightgray', linewidth=0.1)

# Add rectangle
main_bounds = gdf.total_bounds
expansion = 0.1  # 10% expansion on each side

width = main_bounds[2] - main_bounds[0]
height = main_bounds[3] - main_bounds[1]

rect = Rectangle(
    (main_bounds[0] - width * expansion, main_bounds[1] - height * expansion),
    width * (1 + 2 * expansion),
    height * (1 + 2 * expansion),
    linewidth=1.5, edgecolor='red', facecolor='none'
)
ax_inset.add_patch(rect)
ax_inset.axis('off')

# plt.tight_layout()
filename = 'cluster_map_with_inset'
plt.savefig(f"{OUTPUT_PATH_PDF}{filename}.pdf")
plt.savefig(f"{OUTPUT_PATH_PNG}{filename}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{OUTPUT_PATH_PGF}{filename}.pgf")
plt.show()

/var/folders/j3/746qr67x2xj8z46fp8cgvc080000gn/T/ipykernel_8992/2656832758.py:56: UserWarning:

FigureCanvasPgf is non-interactive, and thus cannot be shown

